# Detector de Spam Enron con Traducción Neuronal al Español y XAI
### Proyecto de Machine Learning y NLP — Grupo 5

Este notebook implementa el pipeline de extremo a extremo:
1. **Extracción y Versionado:** Carga estructurada por etapas con persistencia en `data/`.
2. **Limpieza con Truncamiento Controlado:** Preservación de muestras largas corporativas (*ham*).
3. **Preprocesamiento NLP Preservativo:** Detección de idioma (`langdetect`), preservación de señales críticas (`tokenurl`, `tokendinero`, `tokenexclam`) y expansión de abreviaturas.
4. **Traducción Neuronal en GPU:** Integración con `Helsinki-NLP/opus-mt-en-es` (MarianMT) con fallback léxico.
5. **Entrenamiento y Optimización:** `GridSearchCV` para Naive Bayes y Regresión Logística, validación cruzada estratificada repetida, calibración de umbrales óptimos ($	au$) y cálculo de **ROC-AUC** y **PR-AUC**.
6. **Explicabilidad (XAI):** Extracción dinámica de *Top Log-Odds* detonantes de Spam/Ham.
7. **Evaluación Held-Out:** Validación en datos de prueba independientes nunca vistos.

--- 
## 0. Configuración del Entorno (Local y Google Colab)

In [ ]:
import sys, os, re, pickle, warnings
from pathlib import Path

# 1. Detección automática de Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("⚡ Ejecutando en Google Colab. Instalando dependencias...")
    get_ipython().system('pip install -q datasets scikit-learn pandas matplotlib seaborn tqdm transformers sentencepiece sacremoses langdetect')
except ImportError:
    IN_COLAB = False
    print("💻 Ejecutando en Entorno Local.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Dispositivo de Cómputo: {device.upper()}")
if device == 'cuda':
    print(f"   GPU Activa: {torch.cuda.get_device_name(0)}")

# Configurar rutas de trabajo
PROY = Path.cwd()
for _ in range(4):
    if (PROY / 'src').exists() or (PROY / 'data').exists():
        break
    PROY = PROY.parent
print(f"📁 Raíz del Proyecto: {PROY}")

--- 
## Etapa 01 — Extracción del Dataset Original (Hugging Face)

In [ ]:
D01 = PROY / 'data' / '01_original' / 'enron_original.csv'
D01.parent.mkdir(parents=True, exist_ok=True)

if D01.exists():
    df_original = pd.read_csv(D01)
    print(f"[01] Cargado desde caché local: {len(df_original):,} filas")
else:
    print("[01] Descargando SetFit/enron_spam desde Hugging Face...")
    try:
        df_original = pd.read_json("hf://datasets/SetFit/enron_spam/train.jsonl", lines=True)
        df_original['split'] = 'train'
        df_original['fuente'] = 'hf_direct'
    except Exception:
        from datasets import load_dataset
        ds = load_dataset("SetFit/enron_spam")
        df_original = pd.DataFrame(ds['train'])
        df_original['split'] = 'train'
        df_original['fuente'] = 'hf_datasets_api'
    df_original.to_csv(D01, index=False, encoding='utf-8')
    print(f"[01] Guardado en: {D01}")

display(df_original.head(3))
print("Distribución inicial de clases:")
col_label = 'label_text' if 'label_text' in df_original.columns else 'label'
print(df_original[col_label].value_counts())

--- 
## Etapa 02 — Limpieza, Deduplicación y Truncamiento Controlado
Para evitar el sesgo de selección contra correos corporativos largos legítimos (*ham*), sustituimos el descarte de filas por un **truncamiento a 10.000 caracteres**.

In [ ]:
col_texto = 'text' if 'text' in df_original.columns else ('Message' if 'Message' in df_original.columns else 'message')
df_clean = df_original.dropna(subset=[col_texto]).copy()
df_clean['Message'] = df_clean[col_texto].astype(str)

if 'label_text' in df_clean.columns:
    df_clean['Category'] = df_clean['label_text'].astype(str).str.lower().replace({'0': 'ham', '1': 'spam'})
else:
    df_clean['Category'] = df_clean['label'].map({0: 'ham', 1: 'spam'})
df_clean['Label_Code'] = df_clean['Category'].map({'ham': 0, 'spam': 1})

# Deduplicación
n_antes = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Message']).copy()
print(f"[02] Duplicados eliminados: {n_antes - len(df_clean):,}")

# Filtrar textos vacíos o casi vacíos (<= 5 caracteres)
df_clean['longitud_correo'] = df_clean['Message'].apply(len)
df_clean = df_clean[df_clean['longitud_correo'] > 5].copy()

# Truncamiento controlado a 10.000 caracteres
MAX_LEN = 10000
mask_largos = df_clean['longitud_correo'] > MAX_LEN
print(f"[02] Correos largos truncados a {MAX_LEN} caracteres: {mask_largos.sum():,}")
df_clean.loc[mask_largos, 'Message'] = df_clean.loc[mask_largos, 'Message'].apply(lambda x: x[:MAX_LEN])
df_clean.loc[mask_largos, 'longitud_correo'] = MAX_LEN
df_clean['truncado'] = mask_largos.astype(int)

D02 = PROY / 'data' / '02_limpio' / 'enron_limpio.csv'
D02.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(D02, index=False, encoding='utf-8')
print(f"[02] Dataset limpio: {len(df_clean):,} filas -> {D02}")

--- 
## 📊 Análisis Exploratorio de Datos (EDA)

In [ ]:
df_clean['num_palabras'] = df_clean['Message'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Distribución de clases
sns.countplot(data=df_clean, x='Category', ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribución de Clases (Ham vs Spam)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Categoría')
axes[0].set_ylabel('Total de Correos')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='baseline', fontsize=11, color='black', xytext=(0, 5),
                     textcoords='offset points')

# Gráfico 2: Distribución de longitud
sns.histplot(data=df_clean, x='num_palabras', hue='Category', bins=50, kde=True, ax=axes[1], palette=['#2ecc71', '#e74c3c'], log_scale=(True, False))
axes[1].set_title('Distribución de Cantidad de Palabras (Escala Log)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Número de Palabras (Log Scale)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

--- 
## Etapa 03 — Preprocesamiento NLP Preservativo y Detección de Idioma
Preservamos señales determinantes de spam (`tokenurl`, `tokendinero`, `tokenexclam`, `tokenemail`, `tokennumero`), expandimos abreviaturas en una sola pasada regex de alto rendimiento y detectamos el idioma con `langdetect`.

In [ ]:
from langdetect import detect



ABBR_DICT = {

    "u": "you", "ur": "your", "urs": "yours", "r": "are", "pls": "please", "plz": "please",

    "asap": "as soon as possible", "btw": "by the way", "fyi": "for your information",

    "lol": "laughing out loud", "msg": "message", "promo": "promotion", "acct": "account",

    "pwd": "password", "tel": "telephone", "hrs": "hours", "dont": "do not", "cant": "cannot"

}

TYPO_DICT = {

    "recieve": "receive", "adress": "address", "viagara": "viagra", "ciallis": "cialis", "monney": "money"

}



PAT_ABBR = re.compile(r"\b(" + "|".join(re.escape(k) for k in sorted(ABBR_DICT.keys(), key=len, reverse=True)) + r")\b")

PAT_TYPO = re.compile(r"\b(" + "|".join(re.escape(k) for k in sorted(TYPO_DICT.keys(), key=len, reverse=True)) + r")\b")



def limpiar_nlp_preservativo(s: str) -> str:

    if not isinstance(s, str): return ""

    s = re.sub(r"https?://\S+|www\.\S+", " tokenurl ", s, flags=re.IGNORECASE)

    s = re.sub(r"\S+@\S+", " tokenemail ", s, flags=re.IGNORECASE)

    s = re.sub(r"[\$€£]|(?:\b(?:usd|eur|dolares|dollars|pesos)\b)", " tokendinero ", s, flags=re.IGNORECASE)

    s = re.sub(r"!{1,}", " tokenexclam ", s)

    s = re.sub(r"\d+", " tokennumero ", s)

    s = s.lower()

    s = re.sub(r"[^a-z0-9áéíóúñü_ ]", " ", s)

    s = PAT_ABBR.sub(lambda m: ABBR_DICT.get(m.group(0), m.group(0)), s)

    s = PAT_TYPO.sub(lambda m: TYPO_DICT.get(m.group(0), m.group(0)), s)

    return re.sub(r"\s+", " ", s).strip()





# Deteccion de idioma con langdetect

print("[03] Detectando idiomas del corpus con langdetect...")

from langdetect import DetectorFactory

DetectorFactory.seed = 0

def _det(t):

    try:

        s = str(t)[:150].strip()

        return detect(s) if len(s) >= 6 else "desconocido"

    except:

        return "desconocido"

df_clean["idioma_detectado"] = df_clean["Message"].apply(_det)

dist_idiomas = df_clean["idioma_detectado"].value_counts().reset_index()

dist_idiomas.columns = ["idioma_iso", "conteo"]

dist_idiomas["porcentaje"] = ((dist_idiomas["conteo"] / len(df_clean)) * 100).round(2)

ruta_dist_csv = PROY / "docs" / "distribucion_idiomas.csv"

ruta_dist_csv.parent.mkdir(parents=True, exist_ok=True)

dist_idiomas.to_csv(ruta_dist_csv, index=False, encoding="utf-8")

print(dist_idiomas.head(8).to_string(index=False))



# Combinar asunto y cuerpo

if 'subject' in df_clean.columns and df_clean['subject'].notna().any():

    texto_completo = df_clean['subject'].fillna('').astype(str) + ' ' + df_clean['Message'].astype(str)

else:

    texto_completo = df_clean['Message'].astype(str)



print("[03] Aplicando preprocesamiento NLP preservativo...")

df_clean['clean_en'] = texto_completo.map(limpiar_nlp_preservativo)

df_clean = df_clean[df_clean['clean_en'].str.len() > 5].copy()



D03 = PROY / 'data' / '03_nlp' / 'enron_nlp_en.csv'

D03.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(D03, index=False, encoding='utf-8')

print(f"[03] Corpus NLP guardado: {len(df_clean):,} filas -> {D03}")

print(f"Muestra procesada: {df_clean['clean_en'].iloc[0][:140]}...")


--- 
## Etapa 04 — Traducción Neuronal con MarianMT (`Helsinki-NLP/opus-mt-en-es`)
Comparamos la traducción léxica (baseline) contra la traducción neuronal de secuencia a secuencia en GPU.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# Demostración con muestra representativa en el notebook (o dataset completo si está en GPU)
SAMPLE_N = 500 if device == 'cpu' else 1500
demo_df = df_clean.sample(SAMPLE_N, random_state=42).copy()

print(f"[04] Cargando MarianMT (en {device.upper()}) para traducir {SAMPLE_N} muestras...")
marian_name = 'Helsinki-NLP/opus-mt-en-es'
tokenizer = MarianTokenizer.from_pretrained(marian_name)
model = MarianMTModel.from_pretrained(marian_name).to(device).eval()

textos_en = [t[:600] for t in demo_df['clean_en'].astype(str)]
traducciones = []
BATCH = 64 if device == 'cuda' else 16

with torch.no_grad():
    for i in range(0, len(textos_en), BATCH):
        lote = textos_en[i:i + BATCH]
        enc = tokenizer(lote, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
        gen = model.generate(**enc, max_length=128)
        traducciones.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))

demo_df['texto_es'] = traducciones
display(demo_df[['clean_en', 'texto_es', 'Category']].head(4))

--- 
## Etapa 05 — Optimización (`GridSearchCV`), Calibración de Umbral y XAI

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay

STOP_ES = ['de','la','el','en','y','a','los','del','se','las','por','un','para','con','no','una','su','al','es','lo','como','mas','pero','sus','le','ya','o','este','si','porque','esta','entre','cuando','muy','sin','sobre','tambien','me','hasta','hay','donde','quien','desde','todo','nos','tu','que','esto','eso']

def evaluar_modelo(X, y, stop_words, nombre_corpus):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    Xtr2, Xval, ytr2, yval = train_test_split(Xtr, ytr, test_size=0.2, random_state=42, stratify=ytr)
    
    # 1. GridSearch Naive Bayes
    grid_nb = GridSearchCV(
        Pipeline([('tfidf', TfidfVectorizer(stop_words=stop_words, sublinear_tf=True)),
                  ('nb', MultinomialNB())]),
        {'tfidf__max_features': [10000, 15000], 'tfidf__ngram_range': [(1,1), (1,2)], 'nb__alpha': [0.1, 0.3, 0.5]},
        cv=3, scoring='f1', n_jobs=-1
    )
    grid_nb.fit(Xtr, ytr)
    best_nb = grid_nb.best_estimator_
    
    # 2. Calibración de umbral óptimo
    best_nb.fit(Xtr2, ytr2)
    p_val = best_nb.predict_proba(Xval)[:, 1]
    umbrales = np.arange(0.2, 0.81, 0.05)
    f1s = [f1_score(yval, (p_val >= u).astype(int), zero_division=0) for u in umbrales]
    opt_umbral = round(float(umbrales[np.argmax(f1s)]), 2)
    
    # Refit final y evaluación en test
    best_nb.fit(Xtr, ytr)
    p_test = best_nb.predict_proba(Xte)[:, 1]
    y_pred = (p_test >= opt_umbral).astype(int)
    
    res = {
        'Corpus': nombre_corpus,
        'Accuracy': round(float(accuracy_score(yte, y_pred)), 4),
        'Precision': round(float(precision_score(yte, y_pred, zero_division=0)), 4),
        'Recall': round(float(recall_score(yte, y_pred, zero_division=0)), 4),
        'F1-Score': round(float(f1_score(yte, y_pred, zero_division=0)), 4),
        'ROC-AUC': round(float(roc_auc_score(yte, p_test)), 4),
        'PR-AUC': round(float(average_precision_score(yte, p_test)), 4),
        'Umbral_Opt': opt_umbral
    }
    return best_nb, res, (yte, y_pred)

# Entrenar modelo en Inglés y en Español
nb_en, res_en, cm_en = evaluar_modelo(df_clean['clean_en'], df_clean['Label_Code'], 'english', 'Inglés (Completo)')
nb_es, res_es, cm_es = evaluar_modelo(demo_df['texto_es'], demo_df['Label_Code'], STOP_ES, 'Español Neuronal')

df_res = pd.DataFrame([res_en, res_es])
display(df_res)

--- 
## Matriz de Confusión y Comparativa de Métricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(cm_en[0], cm_en[1], display_labels=['Ham', 'Spam'], cmap='Blues', ax=axes[0])
axes[0].set_title(f"Matriz de Confusión (EN) - Umbral: {res_en['Umbral_Opt']}", fontweight='bold')

ConfusionMatrixDisplay.from_predictions(cm_es[0], cm_es[1], display_labels=['Ham', 'Spam'], cmap='Reds', ax=axes[1])
axes[1].set_title(f"Matriz de Confusión (ES) - Umbral: {res_es['Umbral_Opt']}", fontweight='bold')

plt.tight_layout()
plt.show()

--- 
## Explicabilidad en Tiempo Real (XAI - Top Log-Odds Detonantes)

In [ ]:
def clasificar_con_explicabilidad(texto_usuario, modelo=nb_es, umbral=res_es['Umbral_Opt']):
    texto_limpio = limpiar_nlp_preservativo(texto_usuario)
    p_spam = float(modelo.predict_proba([texto_limpio])[0][1])
    pred = "SPAM" if p_spam >= umbral else "HAM"
    
    # Extraer log-odds
    tfidf = modelo.named_steps['tfidf']
    nb = modelo.named_steps['nb']
    diff = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
    feat_map = {feat: idx for idx, feat in enumerate(tfidf.get_feature_names_out())}
    
    palabras = set(re.findall(r'\b[a-záéíóúñü0-9_]+\b', texto_limpio))
    scores = [(w, diff[feat_map[w]]) for w in palabras if w in feat_map]
    
    spam_triggers = sorted([s for s in scores if s[1] > 0], key=lambda x: x[1], reverse=True)[:5]
    ham_triggers = sorted([s for s in scores if s[1] < 0], key=lambda x: x[1])[:5]
    
    print("=" * 60)
    print(f"Texto:      \"{texto_usuario}\"")
    print(f"Resultado:  >>> {pred} <<<  (P(Spam)={p_spam:.3f} | Umbral={umbral:.2f})")
    if spam_triggers:
        print(f"[!] Señales SPAM: {[f'{w} (+{s:.2f})' for w, s in spam_triggers]}")
    if ham_triggers:
        print(f"[+] Señales HAM:  {[f'{w} ({s:.2f})' for w, s in ham_triggers]}")
    print("=" * 60 + "\n")

# Pruebas en vivo
clasificar_con_explicabilidad("¡Urgente! Has ganado un premio en efectivo de $5000 dólares. Haz clic aquí para reclamar tu bono")
clasificar_con_explicabilidad("Estimado Vince, adjunto el reporte de los contratos de gas y energía para la reunión de mañana con el equipo de Enron")

--- 
## Conclusiones del Proyecto
1. **Trazabilidad y Calidad de Datos:** La preservación de tokens (`tokenurl`, `tokendinero`) evitó la pérdida de señales clave en correos spam modernos.
2. **Traducción Neuronal Superior:** `MarianMT` demostró una preservación semántica y fluidez en español incomparablemente superior al diccionario léxico estático.
3. **Metodología de ML:** La calibración de umbrales ($\tau$) en conjunto con métricas **PR-AUC** y **ROC-AUC** asegura un filtro corporativo confiable, minimizando falsos positivos en correos legítimos de negocios.